# Solution: Pareto Optimization of SEI Additives with ALCHEMI

## Control Panel

In [1]:
from pathlib import Path

TOOLKIT_CHECKPOINT = "medium-mpa-0"
TOOLKIT_HEAD = None
TOOLKIT_DEVICE = "auto"
TOOLKIT_DTYPE = "float32"
TOOLKIT_COMPILE_MODEL = False
TOOLKIT_ENABLE_CUEQ = True  # auto-falls back to False if the cu13/cu12 cueq extra is not installed
TOOLKIT_DT = 0.005
TOOLKIT_N_STEPS = 5000
TOOLKIT_FMAX = 0.05
TOOLKIT_FIRE2_MAXSTEP = 0.04
TOOLKIT_D3BJ = None
BATCH_SIZE = 2

SOLUTION_SETTINGS = {
    "min_adsorption_clearance_A": 1.6,
    "vdw_height_scale": 0.66,
    "surface_height_tolerance_A": 1.2,
    "gas_box_A": 20.0,
    "adsorption_site_limit": 3,
    "max_surface_displacement_A": 1.5,
    "frozen_surface_fraction": 0.5,
}

EXAMPLE_SYSTEMS = (
    ("FEC", "li_metal"),
    ("FEC", "passivating"),
    ("VC", "li_metal"),
    ("VC", "passivating"),
    ("EMC", "li_metal"),
    ("EMC", "passivating"),
    ("EC", "li_metal"),
    ("EC", "passivating"),
)

OUTPUT_DIR = Path("outputs")
SUBMISSION_PATH = OUTPUT_DIR / "challenge_submission.csv"
RAW_COMPONENT_ENERGIES_PATH = OUTPUT_DIR / "raw_component_energies.csv"


## Setup

The solution reuses the Part 1 Toolkit backend and keeps challenge-specific geometry/search utilities in `challenge_utils.solution_helpers`.


In [2]:
import os
import sys
import importlib
from importlib.metadata import PackageNotFoundError, version

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "data" / "molecule_manifest.csv").exists():
    candidate = NOTEBOOK_DIR / "challenge-sei"
    if (candidate / "data" / "molecule_manifest.csv").exists():
        NOTEBOOK_DIR = candidate.resolve()
    else:
        raise RuntimeError("Start Jupyter from challenge-sei or from the repository root.")
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

os.environ["ALCHEMI_ALLOW_CACHE_OVERWRITE"] = "1"

REPO_ROOT = NOTEBOOK_DIR.parent
PART1_ROOT = REPO_ROOT / "part-1-batched-adsorption"
if not (PART1_ROOT / "helpers" / "__init__.py").exists():
    raise RuntimeError("Cannot find Part 1 helpers. Keep challenge-sei beside part-1-batched-adsorption.")
sys.path.insert(0, str(PART1_ROOT))

import numpy as np
import pandas as pd

from helpers import (
    ToolkitD3BJConfig,
    check_toolkit_native_api,
    ase_to_atomic_data,
    atomic_data_to_ase,
    display_widgets_grid,
)
from challenge_utils.molecules import build_molecule, known_molecules
from challenge_utils.relaxation_engine import (
    StepByStepRelaxationEngine,
    assemble_pipeline,
    load_mlip,
    resolve_device_and_dtype,
)
from challenge_utils.pareto import dominates, hypervolume_2d, pareto_flags
from challenge_utils.rewards import passivation_score, seeding_score
import challenge_utils.solution_helpers as solution_helpers

importlib.reload(solution_helpers)
from challenge_utils.solution_helpers import (
    SolutionSettings,
    binding_energy_table,
    build_adsorption_surfaces,
    choose_inspection_candidates,
    component_energy_table,
    inspection_widget_rows,
    make_clean_surface_jobs,
    make_combined_jobs,
    make_gas_jobs,
    prepare_challenge_tables,
    relax_structures,
    require_all_converged,
    select_lowest_energy_site_results,
    selected_site_summary,
    surface_summary,
    write_ovito_inspection_structures,
)

SETTINGS = SolutionSettings(**SOLUTION_SETTINGS)

print(f"Challenge folder : {NOTEBOOK_DIR.name}")
print(f"Part 1 helpers   : {PART1_ROOT.relative_to(REPO_ROOT)}")
for pkg in ("ase", "numpy", "pandas", "torch", "nvalchemi-toolkit", "ovito"):
    try:
        print(f"{pkg:<18}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<18}: not installed")


Challenge folder : challenge-sei
Part 1 helpers   : part-1-batched-adsorption
ase               : 3.28.0
numpy             : 2.3.5
pandas            : 3.0.3
torch             : 2.12.0+cu130
nvalchemi-toolkit : 0.1.0
ovito             : not installed


## 1. Load The Challenge Manifests

In [3]:
molecules_df = pd.read_csv("data/molecule_manifest.csv")
custom_manifest_path = Path("data/custom_molecule_manifest.csv")
if custom_manifest_path.exists():
    custom_molecules_df = pd.read_csv(custom_manifest_path)
    molecules_df = pd.concat([molecules_df, custom_molecules_df], ignore_index=True)
    print(f"Loaded {len(custom_molecules_df)} custom/literature molecule row(s).")
else:
    print("No data/custom_molecule_manifest.csv found; using the starter molecule panel.")

if Path("custom_molecules.py").exists():
    import custom_molecules  # noqa: F401  (side effect: registers ase.Atoms geometries)
    print("Imported custom_molecules.py (registered literature geometries).")

if molecules_df["candidate_id"].duplicated().any():
    duplicated = sorted(molecules_df.loc[molecules_df["candidate_id"].duplicated(), "candidate_id"].unique())
    raise RuntimeError(f"Duplicate candidate_id value(s): {duplicated}")

surfaces_df = pd.read_csv("data/surface_manifest.csv")
lookup_df = pd.read_csv("data/class_surface_lookup.csv")
challenge_df, run_systems_df, run_challenge_df = prepare_challenge_tables(
    molecules_df,
    lookup_df,
    EXAMPLE_SYSTEMS,
)

unknown_geometries = sorted(set(challenge_df["candidate_id"]) - set(known_molecules()))
if unknown_geometries:
    raise RuntimeError(
        f"No in-code geometry for: {unknown_geometries}. Register each one in "
        "custom_molecules.py via challenge_utils.molecules.register_molecule "
        "(see custom_molecules_template.py)."
    )

assert set(challenge_df["role"]) == {"baseline", "additive"}
assert {"EC", "EMC"}.issubset(set(challenge_df.loc[challenge_df["role"].eq("baseline"), "candidate_id"]))

print(f"Run systems: {len(run_systems_df)} adsorption system(s); {len(run_challenge_df)} molecule reference(s).")
display(challenge_df[["candidate_id", "role", "molecule_class", "passivating_surface_id", "formula"]])
display(surfaces_df[["surface_id", "role"]])
display(run_systems_df[["candidate_id", "interaction", "surface_id", "role", "formula"]])


No data/custom_molecule_manifest.csv found; using the starter molecule panel.
Run systems: 8 adsorption system(s); 4 molecule reference(s).


,candidate_id,role,molecule_class,passivating_surface_id,formula
0,EC,baseline,carbonate,Li2CO3,C3H4O3
1,EMC,baseline,carbonate,Li2CO3,C4H8O3
2,FEC,additive,fluorinated,LiF,C3H3FO3
3,VC,additive,carbonate,Li2CO3,C3H2O3
4,TMP,additive,phosphate,Li3PO4,C3H9O4P
5,succinonitrile,additive,nitrile,Li3N,C4H4N2


,surface_id,role
0,Li_metal,reactive
1,LiF,passivating
2,Li2CO3,passivating
3,Li3PO4,passivating
4,Li3N,passivating


,candidate_id,interaction,surface_id,role,formula
0,FEC,li_metal,Li_metal,additive,C3H3FO3
1,FEC,passivating,LiF,additive,C3H3FO3
2,VC,li_metal,Li_metal,additive,C3H2O3
3,VC,passivating,Li2CO3,additive,C3H2O3
4,EMC,li_metal,Li_metal,baseline,C4H8O3
5,EMC,passivating,Li2CO3,baseline,C4H8O3
6,EC,li_metal,Li_metal,baseline,C3H4O3
7,EC,passivating,Li2CO3,baseline,C3H4O3


## 2. Building The Relaxation Engine 
We assemble a relxation engine from  Toolkit components.

In [4]:
# Assemble the relaxation engine step by step 
status = check_toolkit_native_api()
print(status["message"])
if not status["available"]:
    raise RuntimeError("ALCHEMI Toolkit native API is not available in this kernel.")

if isinstance(TOOLKIT_D3BJ, dict):
    TOOLKIT_D3BJ = ToolkitD3BJConfig(**TOOLKIT_D3BJ)

# Step 1 - where and in which precision to run.
DEVICE, DTYPE = resolve_device_and_dtype(TOOLKIT_DEVICE, TOOLKIT_DTYPE)
print(f"Step 1 - device={DEVICE}, dtype={TOOLKIT_DTYPE}")

# Step 2 - the MLIP itself.
MLIP = load_mlip(
    TOOLKIT_CHECKPOINT, DEVICE, DTYPE,
    enable_cueq=TOOLKIT_ENABLE_CUEQ, compile_model=TOOLKIT_COMPILE_MODEL,
)
print(f"Step 2 - MACE checkpoint {TOOLKIT_CHECKPOINT!r} loaded")

# Step 3 - the model pipeline the optimizer calls (energy + forces outputs).
MODEL = assemble_pipeline(MLIP, DEVICE, d3bj=TOOLKIT_D3BJ)
print(f"Step 3 - pipeline outputs: {sorted(MODEL.model_config.active_outputs)}; "
      f"D3(BJ): {TOOLKIT_D3BJ is not None}")

# Steps 4-5 (structure -> AtomicData; batched FIRE2 with convergence/neighbor/freeze/NaN
# hooks) execute inside every .relax() call -- see relax_batch()/payload_to_atomic_data().

# Step 6 - the engine object the rest of the workflow consumes.
RELAXATION_ENGINE = StepByStepRelaxationEngine(
    model=MODEL, device=DEVICE, dtype=DTYPE, checkpoint=TOOLKIT_CHECKPOINT,
    dt=TOOLKIT_DT, n_steps=TOOLKIT_N_STEPS, fmax=TOOLKIT_FMAX,
    maxstep=TOOLKIT_FIRE2_MAXSTEP,
)
print(f"Step 6 - relaxation engine ready: {RELAXATION_ENGINE.name}")

Native Toolkit API available: AtomicData, Batch.from_data_list, MACEWrapper, DFTD3ModelWrapper, PipelineModelWrapper, and FIRE2.
Step 1 - device=cuda, dtype=float32
load_mlip: cuequivariance kernels not installed (nvalchemi-toolkit[cu13]); falling back to enable_cueq=False.
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Step 2 - MACE checkpoint 'medium-mpa-0' loaded
Step 3 - pipeline outputs: ['energy', 'forces']; D3(BJ): False
Step 6 - relaxation engine ready: toolkit


### Unit-test the engine before using it

A stretched, rattled water must come back to the textbook geometry
(O–H ≈ 0.96 Å, H–O–H ≈ 104.5°) with forces below the convergence threshold.

In [5]:
from ase.build import molecule as g2_molecule

water = g2_molecule("H2O")
water.positions[1:] *= 1.25                    # stretch both O-H bonds by 25%
water.rattle(stdev=0.02, seed=7)               # break the symmetry slightly
water.set_cell([20.0, 20.0, 20.0])
water.set_pbc(True)
water.center()

_label = "engine_unit_test_water"
reply = RELAXATION_ENGINE.relax(
    [ase_to_atomic_data(water, structure_id="unit_test_water")],
    label=_label,
)
result = reply.atoms[0]
relaxed = atomic_data_to_ase(result)

d_oh = sorted(relaxed.get_distances(0, [1, 2]))
angle_hoh = float(relaxed.get_angle(1, 0, 2))
fmax = float(np.linalg.norm(np.asarray(result.forces).reshape(-1, 3), axis=1).max())

assert result.converged, "engine failed to converge on water"
assert fmax <= TOOLKIT_FMAX + 1e-8, f"forces not converged: fmax={fmax:.3f} eV/A"
assert 0.90 <= d_oh[0] and d_oh[1] <= 1.05, f"O-H bond lengths off: {d_oh}"
assert abs(d_oh[1] - d_oh[0]) < 0.02, f"asymmetric O-H bonds after relaxation: {d_oh}"
assert 95.0 <= angle_hoh <= 115.0, f"H-O-H angle off: {angle_hoh:.1f} deg"
print(
    f"Engine unit test PASSED: O-H = {d_oh[0]:.3f}/{d_oh[1]:.3f} A, "
    f"H-O-H = {angle_hoh:.1f} deg, fmax = {fmax:.3f} eV/A, "
    f"steps = {result.num_optimization_steps}"
)

Engine unit test PASSED: O-H = 0.970/0.970 A, H-O-H = 104.4 deg, fmax = 0.050 eV/A, steps = 111


## 3. Build And Relax The Jobs

As in Part 1, clean slabs are relaxed first. Adsorption starts are then built on the relaxed slabs as a compact `site x orientation x rotation x height` grid.


In [6]:
molecule_atoms = {
    row.candidate_id: build_molecule(row.candidate_id)
    for row in run_challenge_df.itertuples(index=False)
}
surface_meta = surfaces_df.set_index("surface_id")
used_surface_ids = sorted(run_systems_df["surface_id"].unique())

adsorption_surface_atoms = build_adsorption_surfaces(used_surface_ids, settings=SETTINGS)
display(surface_summary(adsorption_surface_atoms, settings=SETTINGS))

gas_jobs = make_gas_jobs(run_challenge_df, molecule_atoms, settings=SETTINGS)
clean_surface_jobs = make_clean_surface_jobs(
    used_surface_ids,
    adsorption_surface_atoms,
    surface_meta,
    settings=SETTINGS,
)
print(f"Gas jobs           : {len(gas_jobs)}")
print(f"Clean-surface jobs : {len(clean_surface_jobs)}")

OUTPUT_DIR.mkdir(exist_ok=True)
gas_results = relax_structures(
    gas_jobs,
    RELAXATION_ENGINE,
    settings=SETTINGS,
    batch_size=BATCH_SIZE,
    label_prefix="solution_sei_gas",
)
clean_surface_results = relax_structures(
    clean_surface_jobs,
    RELAXATION_ENGINE,
    settings=SETTINGS,
    batch_size=BATCH_SIZE,
    label_prefix="solution_sei_clean_surface",
)
require_all_converged(gas_results, label="gas references")
require_all_converged(clean_surface_results, label="clean-surface references")

combined_jobs = make_combined_jobs(
    run_systems_df,
    molecule_atoms,
    surface_meta,
    clean_surface_results,
    settings=SETTINGS,
)
print(f"Combined starts    : {len(combined_jobs)}")

all_combined_results = relax_structures(
    combined_jobs,
    RELAXATION_ENGINE,
    settings=SETTINGS,
    batch_size=BATCH_SIZE,
    label_prefix="solution_sei_combined",
)
combined_results = select_lowest_energy_site_results(all_combined_results)
require_all_converged(combined_results, label="selected combined adsorption systems")
display(selected_site_summary(combined_results))
print("Relaxations complete and converged.")


,surface_id,cell_A,atoms,mobile_surface_atoms,provenance
0,Li2CO3,14.6 x 9.9 x 29.4,144,72,zabuyelite-Li2CO3(001)-COD9008283
1,LiF,17.1 x 12.1 x 26.0,144,72,rocksalt-LiF(100)-physical
2,Li_metal,17.5 x 10.5 x 28.8,90,45,bcc-Li(100)-physical


Gas jobs           : 4
Clean-surface jobs : 3
Combined starts    : 30


W0715 20:55:33.520000 951676 .venv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1853] [1/8] torch._dynamo hit config.recompile_limit (8)
W0715 20:55:33.520000 951676 .venv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1853] [1/8]    function: '_rebuild' (/home/shadeform/ALCHEMI-Bootcamp/challenge-sei/.venv/lib/python3.12/site-packages/nvalchemi/hooks/neighbor_list.py:244)
W0715 20:55:33.520000 951676 .venv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1853] [1/8]    last reason: 1/2: self._neighbor_matrix is None                            # if self._neighbor_matrix is None or self._neighbor_matrix.shape[0] != N:  # nvalchemi/hooks/neighbor_list.py:261 in _rebuild
W0715 20:55:33.520000 951676 .venv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1853] [1/8] User stack trace:
W0715 20:55:33.520000 951676 .venv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1853] [1/8]   File "/home/shadeform/ALCHEMI-Bootcamp/challenge-s

,candidate_id,interaction,surface_id,site_label,start_orientation,energy_eV,fmax_eV_A,surface_max_displacement_A
0,EC,li_metal,Li_metal,center,O-down,-224.165421,0.047994,0.604416
1,EC,passivating,Li2CO3,top_C75,O-down,-1000.113464,0.048827,0.669243
2,EMC,li_metal,Li_metal,center,O-down,-248.125320,0.049564,0.907087
3,EMC,passivating,Li2CO3,center,O-down,-1024.039795,0.049799,1.139054
4,FEC,li_metal,Li_metal,top_Li46,F-down,-225.257263,0.019836,0.870889
5,FEC,passivating,LiF,center,F-down,-751.151917,0.047389,0.088281
6,VC,li_metal,Li_metal,center,O-down,-215.082245,0.048195,0.659192
7,VC,passivating,Li2CO3,top_C75,O-down,-991.410828,0.048018,0.385963


Relaxations complete and converged.


## 4. Compute Binding Energies

Use the same adsorption-energy convention as Part 1: `E_bind = E_surface+species - E_surface - E_species`.


In [7]:
raw_component_energies_df = component_energy_table(
    run_systems_df,
    gas_results,
    clean_surface_results,
    combined_results,
)
raw_component_energies_df.to_csv(RAW_COMPONENT_ENERGIES_PATH, index=False)
print(f"Wrote {RAW_COMPONENT_ENERGIES_PATH}")
display(raw_component_energies_df)

binding_df = binding_energy_table(run_challenge_df, raw_component_energies_df)
display(binding_df[["candidate_id", "role", "E_bind_Li_eV", "E_bind_passivating_eV"]])


Wrote outputs/raw_component_energies.csv


,candidate_id,interaction,surface_id,E_surface_species_eV,E_surface_eV,E_species_eV,selected_site_label,selected_start_orientation
0,FEC,li_metal,Li_metal,-225.257263,-161.196442,-61.295509,top_Li46,F-down
1,FEC,passivating,LiF,-751.151917,-688.896545,-61.295509,center,F-down
2,VC,li_metal,Li_metal,-215.082245,-161.196442,-52.123497,center,O-down
3,VC,passivating,Li2CO3,-991.410828,-938.285278,-52.123497,top_C75,O-down
4,EMC,li_metal,Li_metal,-248.125320,-161.196442,-84.991806,center,O-down
5,EMC,passivating,Li2CO3,-1024.039795,-938.285278,-84.991806,center,O-down
6,EC,li_metal,Li_metal,-224.165421,-161.196442,-60.814941,center,O-down
7,EC,passivating,Li2CO3,-1000.113464,-938.285278,-60.814941,top_C75,O-down


,candidate_id,role,E_bind_Li_eV,E_bind_passivating_eV
0,EC,baseline,-2.154037,-1.013245
1,EMC,baseline,-1.937073,-0.762711
2,FEC,additive,-2.765312,-0.959862
3,VC,additive,-1.762306,-1.002052


## 5. Inspect Relaxed Geometries

Write the selected relaxed adsorption structures as EXTXYZ and show them with the Part 1 OVITO widget helper when available.


In [8]:
OVITO_STRUCTURE_DIR = OUTPUT_DIR / "ovito_structures"
inspection_df = write_ovito_inspection_structures(combined_results, output_dir=OVITO_STRUCTURE_DIR)
INSPECT_CANDIDATE_IDS = choose_inspection_candidates(binding_df)

print(f"Wrote {len(inspection_df)} OVITO structure file(s) to {OVITO_STRUCTURE_DIR}")
print("Inspecting:", ", ".join(INSPECT_CANDIDATE_IDS))
display(inspection_df[inspection_df["candidate_id"].isin(INSPECT_CANDIDATE_IDS)])

try:
    display_widgets_grid(
        inspection_widget_rows(inspection_df, INSPECT_CANDIDATE_IDS),
        width="390px",
        height="310px",
        show_cell=False,
    )
except Exception as exc:
    print(f"OVITO widget display unavailable: {type(exc).__name__}: {exc}")
    display(inspection_df[["candidate_id", "interaction", "surface_id", "structure_path"]])


Wrote 8 OVITO structure file(s) to outputs/ovito_structures
Inspecting: EC, EMC, FEC


,candidate_id,interaction,surface_id,energy_eV,converged,structure_path
0,EC,li_metal,Li_metal,-224.165421,True,outputs/ovito_structures/EC_li_metal_Li_metal....
1,EC,passivating,Li2CO3,-1000.113464,True,outputs/ovito_structures/EC_passivating_Li2CO3...
2,EMC,li_metal,Li_metal,-248.125320,True,outputs/ovito_structures/EMC_li_metal_Li_metal...
3,EMC,passivating,Li2CO3,-1024.039795,True,outputs/ovito_structures/EMC_passivating_Li2CO...
4,FEC,li_metal,Li_metal,-225.257263,True,outputs/ovito_structures/FEC_li_metal_Li_metal...
5,FEC,passivating,LiF,-751.151917,True,outputs/ovito_structures/FEC_passivating_LiF.e...


## 6. Compute Reward Scores

The challenge rubric rewards moderate Li-metal binding for SEI seeding and weak binding on the passivating proxy surface.


In [9]:
scored_df = binding_df.copy()
scored_df["seeding_score"] = scored_df["E_bind_Li_eV"].map(seeding_score)
scored_df["passivation_score"] = scored_df["E_bind_passivating_eV"].map(passivation_score)

display(scored_df[["candidate_id", "role", "seeding_score", "passivation_score"]])


,candidate_id,role,seeding_score,passivation_score
0,EC,baseline,0.704969,0.409651
1,EMC,baseline,0.885773,0.767556
2,FEC,additive,0.195573,0.485912
3,VC,additive,1.000000,0.425640


## 7. Pareto Front And Hypervolume

Treat both scores as objectives to maximize. Hypervolume improvement is measured against the baseline `EC`/`EMC` front with reference point `(0, 0)`.


In [10]:
final_df = scored_df.copy()
points = list(zip(final_df["seeding_score"], final_df["passivation_score"]))
final_df["is_pareto"] = pareto_flags(points)

baseline_points = list(zip(
    final_df.loc[final_df["role"].eq("baseline"), "seeding_score"],
    final_df.loc[final_df["role"].eq("baseline"), "passivation_score"],
))
baseline_hv = hypervolume_2d(baseline_points)

final_df["hypervolume_improvement"] = [
    0.0 if row.role == "baseline"
    else hypervolume_2d([*baseline_points, (row.seeding_score, row.passivation_score)]) - baseline_hv
    for row in final_df.itertuples(index=False)
]

print(f"Baseline hypervolume: {baseline_hv:.4f}")
display(final_df.sort_values("hypervolume_improvement", ascending=False))


Baseline hypervolume: 0.6799


,candidate_id,role,molecule_class,passivating_surface_id,E_bind_Li_eV,E_bind_passivating_eV,seeding_score,passivation_score,is_pareto,hypervolume_improvement
3,VC,additive,carbonate,Li2CO3,-1.762306,-1.002052,1.000000,0.425640,True,0.04862
0,EC,baseline,carbonate,Li2CO3,-2.154037,-1.013245,0.704969,0.409651,False,0.00000
1,EMC,baseline,carbonate,Li2CO3,-1.937073,-0.762711,0.885773,0.767556,True,0.00000
2,FEC,additive,fluorinated,LiF,-2.765312,-0.959862,0.195573,0.485912,False,0.00000


## 8. Select Your Additive And Submit

Mark exactly one additive as selected: the additive with the largest hypervolume improvement.


In [11]:
submission = final_df.copy()
additives = submission[submission["role"].eq("additive")].copy()
if additives.empty:
    raise RuntimeError("No additive rows are available to select.")

# Rank additives by hypervolume improvement, then by non-dominated (is_pareto);
# a stable sort keeps the choice deterministic.
selected_id = (
    additives
    .sort_values(
        ["hypervolume_improvement", "is_pareto"],
        ascending=[False, False],
        kind="stable",
    )
    .iloc[0]["candidate_id"]
)
submission["selected"] = submission["candidate_id"].eq(selected_id)

print(f"Selected additive: {selected_id}")
display(submission[[
    "candidate_id", "role", "seeding_score", "passivation_score",
    "is_pareto", "hypervolume_improvement", "selected",
]].sort_values("hypervolume_improvement", ascending=False))


Selected additive: VC


,candidate_id,role,seeding_score,passivation_score,is_pareto,hypervolume_improvement,selected
3,VC,additive,1.000000,0.425640,True,0.04862,True
0,EC,baseline,0.704969,0.409651,False,0.00000,False
1,EMC,baseline,0.885773,0.767556,True,0.00000,False
2,FEC,additive,0.195573,0.485912,False,0.00000,False


In [12]:
required_columns = [
    "candidate_id", "role", "molecule_class", "passivating_surface_id",
    "E_bind_Li_eV", "E_bind_passivating_eV", "seeding_score",
    "passivation_score", "is_pareto", "hypervolume_improvement", "selected",
]
missing = [column for column in required_columns if column not in submission.columns]
if missing:
    raise RuntimeError(f"Submission is missing required columns: {missing}")
if int(submission["selected"].sum()) != 1:
    raise RuntimeError("Exactly one row must be selected.")

OUTPUT_DIR.mkdir(exist_ok=True)
submission[required_columns].to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}")
display(submission[required_columns])


Wrote outputs/challenge_submission.csv


,candidate_id,role,molecule_class,passivating_surface_id,E_bind_Li_eV,E_bind_passivating_eV,seeding_score,passivation_score,is_pareto,hypervolume_improvement,selected
0,EC,baseline,carbonate,Li2CO3,-2.154037,-1.013245,0.704969,0.409651,False,0.00000,False
1,EMC,baseline,carbonate,Li2CO3,-1.937073,-0.762711,0.885773,0.767556,True,0.00000,False
2,FEC,additive,fluorinated,LiF,-2.765312,-0.959862,0.195573,0.485912,False,0.00000,False
3,VC,additive,carbonate,Li2CO3,-1.762306,-1.002052,1.000000,0.425640,True,0.04862,True


## References And Further Reading

- NVIDIA [ALCHEMI Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/) for `AtomicData`, `Batch`, model wrappers, and Toolkit dynamics.
- Batatia et al., [MACE: Higher Order Equivariant Message Passing Neural Networks for Fast and Accurate Force Fields](https://openreview.net/forum?id=YPpSngE-ZU), NeurIPS 2022.
- Larsen et al., [The Atomic Simulation Environment - a Python library for working with atoms](https://doi.org/10.1088/1361-648X/aa680e), J. Phys.: Condens. Matter 2017.
- Stukowski, [Visualization and analysis of atomistic simulation data with OVITO - the Open Visualization Tool](https://doi.org/10.1088/0965-0393/18/1/015012), Modelling Simul. Mater. Sci. Eng. 2010.
- Leung et al., [Stability of Solid Electrolyte Interphase Components on Lithium Metal and Reactive Anode Material Surfaces](https://doi.org/10.1021/acs.jpcc.5b11719), J. Phys. Chem. C 2016; examples use large periodic SEI/Li cells and matching slab-interface references.
- Chanussot et al., [The Open Catalyst 2020 Dataset and Community Challenges](https://doi.org/10.1021/acscatal.0c04525), ACS Catalysis 2021; summarizes common slab-adsorbate setup, adsorption-energy references, vacuum, and fixed subsurface atoms.
- Shi et al., [Review on modeling of the anode solid electrolyte interphase (SEI) for lithium-ion batteries](https://www.nature.com/articles/s41524-018-0064-0), npj Computational Materials 2018.
- Xu et al., [A review on electrolyte additives for lithium-ion batteries](https://www.sciencedirect.com/science/article/pii/S0378775306017538), J. Power Sources 2007.
- Balakrishnan et al., [Electrolyte additives for improved lithium-ion battery performance and overcharge protection](https://www.sciencedirect.com/science/article/pii/S2451910320300089), 2020.
- Li et al., [A Review of Solid Electrolyte Interphases on Lithium Metal Anode](https://pmc.ncbi.nlm.nih.gov/articles/PMC5063117/), Advanced Science 2016.
- [Insights into the efficient roles of solid electrolyte interphase derived from vinylene carbonate additive in rechargeable batteries](https://www.sciencedirect.com/science/article/abs/pii/S1572665722001187), 2022.
- Zhang et al., [Reduction Mechanism of Fluoroethylene Carbonate for Stable Solid-Electrolyte Interphase Film on Silicon Anode](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film), ChemSusChem 2013.
- Lee et al., [The Sabatier Principle in Electrocatalysis: Basics, Limitations, and Extensions](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full), Frontiers in Energy Research 2021.
- Aich et al., [Determination of thermodynamic parameters in adsorption studies: a review](https://link.springer.com/article/10.1007/s11696-025-04218-x), Chemical Papers 2025.
- [Hypervolume bibliography](https://hypervolume.org/bibliography.html) for Pareto hypervolume indicator references.
